# Optimization & Data Management

OPTIMIZE, VACUUM, partitioning, Z-ORDER, Liquid Clustering, and query plan analysis. Practical techniques for Delta table maintenance and performance.

| Training Block | Duration | Type |
|---|---|---|
| Optimization — Demo | 50 min | Demo |
| Optimization — Workshop | 30 min | Hands-on |

**Prerequisites:** 02 — Delta Lake Advanced

## Learning Objectives

After completing this module you will be able to:

- **Diagnose** the small file problem and apply `OPTIMIZE` for compaction
- **Implement** partitioning strategies and understand when NOT to partition
- **Apply** Z-ORDER and Liquid Clustering for data skipping optimization
- **Manage** table lifecycle with `VACUUM` (retention, safety considerations)
- **Analyze** query plans with `EXPLAIN` to identify performance bottlenecks

## Setup

Environment configuration and database initialization for the optimization demos.

In [0]:
%run ../../setup/00_setup

### Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta

# Display user context
display(
    spark.createDataFrame([
        (CATALOG, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA)
    ], ['catalog', 'bronze_schema', 'silver_schema', 'gold_schema'])
)

# Set catalog and schema as default
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

## Optimization Techniques

File compaction (OPTIMIZE), data skipping (Z-ORDER), partitioning strategies, and Liquid Clustering — key techniques for query performance on large Delta tables.

**Theoretical Introduction:**

As data grows, query performance can degrade due to several factors:
- **Small Files Problem**: Too many small files increase metadata overhead
- **Data Layout**: Data not organized for common query patterns
- **Predicate Pushdown Inefficiency**: Scanning more data than necessary

Delta Lake provides several optimization techniques:

| Technique | Description | When to Use |
|-----------|-------------|-------------|
| **OPTIMIZE** | Compacts small files into larger ones | After many small writes |
| **Partitioning** | Physical data separation by column values | Low-cardinality filter columns |
| **Z-ORDER** | Co-locates related data for better pruning | Frequently filtered columns |
| **Liquid Clustering** | Modern alternative to partitioning + Z-ORDER | New tables (recommended) |

<img src="../../../assets/images/6653c397bbc24993975c4fc11a356ab6.png" width="800">

**Optimization — Syntax Reference**

| Command | Syntax | Description |
|---------|--------|-------------|
| `OPTIMIZE` | `OPTIMIZE table` | Compact small files (target ~1 GB per file) |
| `OPTIMIZE + ZORDER` | `OPTIMIZE table ZORDER BY (col1, col2)` | Compact + co-locate data (max 4 cols) |
| `VACUUM` | `VACUUM table RETAIN 168 HOURS` | Remove unreferenced files (default: 7 days) |
| `VACUUM 0H` | `SET spark.databricks.delta.retentionDurationCheck.enabled = false; VACUUM t RETAIN 0 HOURS` | Removes ALL history |
| `DESCRIBE DETAIL` | `DESCRIBE DETAIL table` | Show file count, size, location, clustering |
| `DESCRIBE HISTORY` | `DESCRIBE HISTORY table` | Show all Delta versions with operation info |

### Example: The Small Files Problem

**Objective:** Demonstrate how many small files impact performance and how OPTIMIZE solves it

The "small files problem" occurs when:
- Streaming jobs write many small files
- Frequent small batch inserts
- High-concurrency writes

This leads to:
- Increased metadata overhead
- Slower query performance
- Higher storage costs (metadata per file)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.small_files_demo")

In [0]:
# Create a table with many small files (simulating streaming ingestion)
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.small_files_demo (
    id INT,
    data STRING,
    created_at TIMESTAMP
) USING DELTA
TBLPROPERTIES (
    delta.autoOptimize.optimizeWrite = false,
    delta.autoOptimize.autoCompact = false
)
""")

In [0]:
# Insert data in many small batches (simulating streaming)
from pyspark.sql.functions import lit, current_timestamp
import random
import string

print("Inserting 500 small batches to simulate streaming ingestion...")

In [0]:
%sql 

DESCRIBE DETAIL small_files_demo

In [0]:
from pyspark.sql.functions import lit, expr, current_timestamp
import random

In [0]:

# 1. Configuration
total_files = 5000
rows_per_file = 2  # Average 2 records per file
total_rows = total_files * rows_per_file

In [0]:
# 2. Generate data in memory (no Python loop!)
df = (
    spark.range(0, total_rows)
    .withColumn("id", lit(random.randint(1, 100)))
    .withColumn("data", expr("uuid()"))
    .withColumn("created_at", current_timestamp())
)

In [0]:
# 3. Write with forced number of files
df.repartition(total_files).write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.small_files_demo")

In [0]:
print(f"Done! Created {total_files} small files in a single transaction.")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.small_files_demo"))

In [0]:
# Check the number of files BEFORE optimization
before_optimize = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.small_files_demo")
display(before_optimize.select("numFiles", "sizeInBytes"))
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.small_files_demo"))

In [0]:
# Run OPTIMIZE to compact small files
optimize_result = spark.sql(f"""
    OPTIMIZE {CATALOG}.{BRONZE_SCHEMA}.small_files_demo
""")

display(optimize_result)

In [0]:
# ⚠ DANGER ZONE: VACUUM with 0 hours retention
# This disables the safety check and immediately deletes ALL old file versions.
# In production, NEVER use RETAIN 0 HOURS — use default 168 hours (7 days) or more!
# Files removed by VACUUM cannot be recovered. Time travel will stop working.
# We use this here ONLY because this is a training demo with disposable data.
spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled = false")

vacuum_result = spark.sql(f"""
    VACUUM {CATALOG}.{BRONZE_SCHEMA}.small_files_demo RETAIN 0 HOURS
""")

# Re-enable the safety check immediately
spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled = true")
print("VACUUM complete. Safety check re-enabled.")


In [0]:
# Check the number of files AFTER optimization
after_optimize = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.small_files_demo")
display(after_optimize.select("numFiles", "sizeInBytes"))

### Example: Partitioning

**Objective:** Demonstrate how partitioning improves query performance through partition pruning

| Syntax | Description |
|---|---|
| `PARTITIONED BY (col)` | Physically separates data into directories by column value |
| `PARTITIONED BY (col1, col2)` | Multi-column partitioning — each combination gets its own directory |
| `ALTER TABLE ... ADD PARTITION` | Adds a new partition to an existing table |
| `SHOW PARTITIONS table` | Lists all existing partitions |

Partitioning physically separates data into directories based on column values. This enables:
- **Partition Pruning**: Skip entire partitions that don't match query filters
- **Parallel Processing**: Process partitions independently

**Best Practices:**
- Use low-cardinality columns (date, country, status)
- Avoid over-partitioning (too many small partitions)
- Aim for 1GB+ per partition

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.orders_partitioned")

In [0]:
# Create a partitioned table
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.orders_partitioned (
    order_id STRING,
    customer_id STRING,
    product_id STRING,
    order_date DATE,
    amount DOUBLE,
    status STRING
) 
USING DELTA
PARTITIONED BY (order_date,status)
""")

In [0]:
# Insert sample data across multiple dates
from datetime import date, timedelta

orders_data = []
base_date = date(2024, 1, 1)

for day_offset in range(30):  # 30 days of data
    order_date = base_date + timedelta(days=day_offset)
    for i in range(100):  # 100 orders per day
        orders_data.append((
            f"ORD-{day_offset:02d}-{i:04d}",
            f"CUST{i % 50:04d}",
            f"PROD{i % 20:03d}",
            order_date,
            50 + (i * 2.5),
            "completed" if i % 3 != 0 else "pending"
        ))

orders_df = spark.createDataFrame(orders_data, 
    ["order_id", "customer_id", "product_id", "order_date", "amount", "status"])

orders_df.write.format("delta").mode("append").partitionBy("order_date","status") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.orders_partitioned")

print(f"Inserted {len(orders_data)} orders across 30 days")

In [0]:
# Check partitioning structure
display(spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.orders_partitioned"))

In [0]:
# Query with partition filter - only scans relevant partitions
# Check the Spark UI to see partition pruning in action
result = spark.sql(f"""
    SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.orders_partitioned
    WHERE order_date = '2024-01-15'
""")

print("Query for single date (should scan only 1 partition):")
display(result)

### Example: Z-ORDER (Data Skipping)

**Objective:** Demonstrate Z-ORDER for multi-dimensional clustering

| Command | Signature | Description |
|---|---|---|
| `OPTIMIZE ... ZORDER BY` | `OPTIMIZE table ZORDER BY (col1, col2)` | Compacts files AND re-orders data using Z-order curve for data skipping |
| `OPTIMIZE` (basic) | `OPTIMIZE table` | Compacts small files only — no Z-ORDER |

Z-ORDER is a multi-dimensional clustering technique that co-locates related data within files. This enables **data skipping** - reading only relevant files based on min/max statistics.

**When to Use:**
- Columns frequently in WHERE clauses
- High-cardinality columns (customer_id, product_id)
- Up to 4 columns (effectiveness decreases with more)

**How it Works:**
- Reorganizes data within files using Z-order curve
- Maintains min/max statistics per file
- Query engine skips files that don't match predicates

<img src="../../../assets/images/8ddc3a9209e145cf8edec763d45344d7.png" width="800">

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo")

In [0]:
# Create a table for Z-ORDER demonstration with auto-optimization disabled
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo (
    sale_id STRING,
    customer_id STRING,
    product_id STRING,
    store_id STRING,
    sale_date DATE,
    amount DOUBLE,
    quantity INT
) USING DELTA
TBLPROPERTIES (
    delta.autoOptimize.optimizeWrite = false,
    delta.autoOptimize.autoCompact = false
)
""")

In [0]:
# Insert sample data
from datetime import date
import random

sales_data = []
for i in range(100000):  # 100K records
    sales_data.append((
        f"SALE-{i:08d}",
        f"CUST{random.randint(1, 1000):04d}",
        f"PROD{random.randint(1, 500):03d}",
        f"STORE{random.randint(1, 50):02d}",
        date(2024, random.randint(1, 12), random.randint(1, 28)),
        random.uniform(10, 500),
        random.randint(1, 10)
    ))

sales_df = spark.createDataFrame(
    sales_data, 
    ["sale_id", "customer_id", "product_id", "store_id", "sale_date", "amount", "quantity"]
)

sales_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo"
)
display(sales_df)

In [0]:
# Check file statistics BEFORE Z-ORDER
display(spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo"))

In [0]:
#Queries filtering before Z-Order
result = spark.sql(f"""
    SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo
    WHERE customer_id = 'CUST0393' AND product_id = 'PROD259' --CUST0592	PROD011
""")

display(result)

In [0]:
# Apply Z-ORDER on frequently filtered columns
# In this case: customer_id and product_id are common filter columns
zorder_result = spark.sql(f"""
    OPTIMIZE {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo
    ZORDER BY (customer_id, product_id)
""")

display(zorder_result)

In [0]:
# Example query that benefits from Z-ORDER
result = spark.sql(f"""
    SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo
     WHERE customer_id = 'CUST0393' AND product_id = 'PROD259' --CUST0592	PROD011
""")

print("Query with Z-ORDER optimized columns (check Spark UI for data skipping):")
display(result)

### Example: Liquid Clustering

**Objective:** Demonstrate Liquid Clustering as a modern alternative to partitioning and Z-ORDER

| Syntax | Description |
|---|---|
| `CLUSTER BY (col1, col2)` | Defines clustering columns at table creation |
| `CLUSTER BY AUTO` | Databricks automatically selects optimal clustering columns |
| `ALTER TABLE ... CLUSTER BY (col)` | Changes clustering columns without rewriting data |
| `OPTIMIZE table` | Applies Liquid Clustering incrementally (no ZORDER needed) |

Liquid Clustering is Databricks' latest optimization technique that combines the benefits of partitioning and Z-ORDER while being easier to manage:

**Key Benefits:**
- **Automatic**: Databricks manages data layout automatically
- **Adaptive**: Adjusts to changing query patterns over time
- **Flexible**: Can change clustering columns without rewriting data
- **Incremental**: Works incrementally with each OPTIMIZE
- **Simpler**: No need to choose between partitioning and Z-ORDER

**When to Use:**
- New tables (recommended default)
- Tables with evolving query patterns
- When you're unsure about optimal partitioning strategy
**Automatic Liquid Clustering (GA June 2025):**
- Use `CLUSTER BY AUTO` — Databricks automatically selects the best clustering columns based on query patterns
- No need to manually choose columns — the system learns from workload
- Combine with Predictive Optimization for fully automated table maintenance

**Automatic Liquid Clustering (GA June 2025):**
- Use `CLUSTER BY AUTO` — Databricks automatically selects the best clustering columns based on query patterns
- No need to manually choose columns — the system learns from workload
- Combine with Predictive Optimization for fully automated table maintenance

**Automatic Liquid Clustering (GA June 2025):**
- Use `CLUSTER BY AUTO` — Databricks automatically selects the best clustering columns based on query patterns
- No need to manually choose columns — the system learns from workload
- Combine with Predictive Optimization for fully automated table maintenance

**Liquid Clustering — Syntax Reference**

| Operation | Syntax |
|-----------|--------|
| Create with clustering | `CREATE TABLE t (...) CLUSTER BY (col1, col2)` |
| Add to existing table | `ALTER TABLE t CLUSTER BY (col1, col2)` |
| Change columns | `ALTER TABLE t CLUSTER BY (col3)` — no data rewrite needed |
| Remove clustering | `ALTER TABLE t CLUSTER BY NONE` |
| Trigger compaction | `OPTIMIZE t` — applies liquid clustering automatically |
| Check clustering | `DESCRIBE DETAIL t` — see `clusteringColumns` field |

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering")

In [0]:
# Create a table with Liquid Clustering, auto-optimization disabled, and Predictive Optimization enabled
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering (
    sale_id STRING,
    customer_id STRING,
    product_id STRING,
    region STRING,
    sale_date DATE,
    amount DOUBLE,
    quantity INT
) 
USING DELTA
CLUSTER BY (customer_id, region)
TBLPROPERTIES (
    delta.autoOptimize.optimizeWrite = false,
    delta.autoOptimize.autoCompact = false
)
""")

In [0]:
spark.sql(f""" ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering SET TBLPROPERTIES ('spark.databricks.sql.predictiveOptimization.enabled'='true'); """)

In [0]:
from pyspark.sql.functions import col, rand, lit, concat, lpad, element_at, array, date_add, to_date, round

# 1. Configuration
target_files = 5000       # We want 5000 files
rows_per_file = 10        # 10 records per file
total_rows = target_files * rows_per_file # Total 50,000 records

# Array of regions for random selection
regions_list = array([lit(x) for x in ['North', 'South', 'East', 'West', 'Central']])

# 2. Data generation (no for loop!)
df = spark.range(0, total_rows).withColumnRenamed("id", "idx") \
    .withColumn("sale_id", concat(lit("SALE-"), lpad(col("idx"), 8, "0"))) \
    .withColumn("customer_id", concat(lit("CUST"), lpad((rand() * 500 + 1).cast("int"), 4, "0"))) \
    .withColumn("product_id", concat(lit("PROD"), lpad((rand() * 200 + 1).cast("int"), 3, "0"))) \
    .withColumn("region", element_at(regions_list, (rand() * 5 + 1).cast("int"))) \
    .withColumn("sale_date", date_add(to_date(lit("2024-01-01")), (rand() * 364).cast("int"))) \
    .withColumn("amount", round(rand() * 490 + 10, 2)) \
    .withColumn("quantity", (rand() * 10 + 1).cast("int")) \
    .drop("idx") # Remove helper column

# 3. Write - repartition is key
# Create table with Liquid Clustering enabled (if not exists)
# or append to existing one.
df.repartition(target_files).write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering")

print(f"Done! Inserted {total_rows} records in {target_files} small files.")

In [0]:
df = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering")
display(df)

In [0]:
# Change clustering columns on an existing table (no data rewrite needed!)
# CLUSTER BY AUTO lets Databricks automatically choose optimal clustering columns
spark.sql(f"""
ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering
CLUSTER BY AUTO
""")

In [0]:
# OPTIMIZE automatically applies Liquid Clustering
# No need to specify ZORDER - it's built into the table definition!
optimize_result = spark.sql(f"""
    OPTIMIZE {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering
""")

display(optimize_result)

In [0]:
# Check clustering information
display(spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering"))

In [0]:
# Queries filtering by clustering columns are automatically optimized
result = spark.sql(f"""
    SELECT region, COUNT(*) as sales_count, SUM(amount) as total_amount
    FROM {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering
    WHERE customer_id LIKE 'CUST00%' AND region = 'North'
    GROUP BY region
""")

display(result)

**Comparison: Partitioning vs Z-ORDER vs Liquid Clustering**

| Feature | Partitioning | Z-ORDER | Liquid Clustering |
|---------|-------------|---------|-------------------|
| When to choose | Low-cardinality columns | High-cardinality filter columns | General purpose (recommended) |
| Data layout | Directory per partition | Co-located in files | Automatic clustering |
| Schema change | Requires rewrite | Easy to change | Easy to change |
| Maintenance | Manual | Manual OPTIMIZE | Automatic with OPTIMIZE |
| Best for | Date/Region filters | Multi-column filters | Evolving workloads |

## Query Plan Analysis with EXPLAIN

Use `EXPLAIN` to understand how Spark executes a query — identify full scans, shuffles, and data skipping.

| Command | Description |
|---|---|
| `EXPLAIN query` | Basic physical plan |
| `EXPLAIN FORMATTED query` | Detailed plan with metrics |
| `EXPLAIN EXTENDED query` | Logical + physical + optimized plan |

**What to look for:**
- [OK] **PartitionFilters** — partition pruning is active
- [OK] **DataFilters / PushedFilters** — file-level data skipping
- [OK] **BroadcastHashJoin** — small table broadcast (no shuffle)
- [!] **SortMergeJoin** — both sides shuffled (expensive)
- [!] **Scan** without filters — full table scan

In [0]:
# EXPLAIN on a Z-ORDERed table (should show DataFilters / data skipping)
spark.sql(f"""
    EXPLAIN FORMATTED
    SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo
    WHERE customer_id = 'CUST0393' AND product_id = 'PROD259'
""").show(truncate=False)

In [0]:
# Compare: EXPLAIN on a Liquid Clustered table
spark.sql(f"""
    EXPLAIN FORMATTED
    SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering
    WHERE product_id = 'PROD042'
""").show(truncate=False)

In [0]:
# EXPLAIN with JOIN — check for BroadcastHashJoin vs SortMergeJoin
spark.sql(f"""
    EXPLAIN FORMATTED
    SELECT o.*, c.first_name, c.last_name
    FROM {CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo o
    JOIN {CATALOG}.{BRONZE_SCHEMA}.customers c
      ON o.customer_id = c.customer_id
    WHERE o.product_id = 'PROD259'
""").show(truncate=False)

> **SKIP IF SHORT ON TIME** — Data skew diagnosis is important but can be covered briefly if time is tight.

## Data Skew & Distribution Problems

**Data skew** occurs when data is unevenly distributed across partitions. Instead of each partition having roughly the same number of rows, one or more partitions end up with significantly more data ("hot partitions"). This is one of the most common and impactful performance problems in distributed data processing.

<img src="../../../assets/images/eef58dd32657427eb8f4f2fdacff9b39.png" width="800">

### Common Causes

| Cause | Example | Impact |
|-------|---------|--------|
| **Skewed join keys** | Joining on `country` where 80% of customers are from one country | One executor handles most of the join |
| **NULL values in join/group keys** | `customer_id IS NULL` for anonymous orders | All NULLs land on one partition |
| **Hot keys in groupBy** | `groupBy("product_category")` where "Electronics" has 90% of sales | One partition aggregates most data |
| **Uneven source files** | One file is 10 GB, others are 100 MB | One task reads disproportionate data |
| **Time-based partitioning** | `PARTITION BY (order_date)` when most orders are from last month | Recent partition is orders of magnitude larger |

### Detecting Data Skew

**Symptoms:**
- One stage takes much longer than others in Spark UI
- One task in a stage uses much more memory/time than siblings
- `SpillToMemory` or `SpillToDisk` metrics appear for one task
- OOM (Out of Memory) errors on specific executors

In [0]:
# Diagnostic: check partition size distribution
# df.groupBy(spark_partition_id()).count().orderBy("count", ascending=False).show()

# Diagnostic: check key distribution for join/group columns
# df.groupBy("join_key_column").count().orderBy("count", ascending=False).show(20)

# In Databricks SQL:
# SELECT join_key, COUNT(*) as cnt FROM table GROUP BY join_key ORDER BY cnt DESC LIMIT 20

### Solution 1: Adaptive Query Execution (AQE) — Automatic

AQE is enabled by default in Databricks. It detects skew at runtime and automatically:
- Splits skewed partitions into smaller sub-partitions
- Adjusts shuffle partition count based on data volume
- Converts sort-merge joins to broadcast joins when beneficial

> **Pro Tip:** AQE is a Spark 3.x feature enabled by default in Databricks. Know that it can dynamically adjust partition counts and handle skewed joins automatically.

In [0]:
# AQE is ON by default in Databricks. Verify and tune key settings:
# spark.conf.get("spark.sql.adaptive.enabled")                        # true
# spark.conf.get("spark.sql.adaptive.skewJoin.enabled")               # true
# spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionFactor") # 5 (5x median = skewed)
# spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes")  # 256MB

# No manual intervention needed — AQE handles most skew scenarios automatically

### Solution 2: Salting — Manual Technique for Extreme Skew

When AQE is insufficient (100:1+ key imbalance), **salting** splits hot keys artificially by appending a random suffix to distribute data across more partitions.

In [0]:
from pyspark.sql.functions import concat, lit, floor, rand, explode, array

# Salting pattern — split hot join key into N sub-keys
num_salts = 10

# Large (skewed) table: add random salt suffix
# df_large_salted = df_large.withColumn(
#     "salted_key", concat("join_key", lit("_"), floor(rand() * num_salts).cast("int"))
# )

# Small (lookup) table: explode to match all salt values
# df_small_salted = df_small.withColumn(
#     "salted_key",
#     explode(array([concat("join_key", lit(f"_{i}")) for i in range(num_salts)]))
# )

# Join on salted key — data distributed evenly across num_salts partitions
# result = df_large_salted.join(df_small_salted, "salted_key")

### Solution 3: Broadcast Join — Small Table Optimization

If one side of the join is small enough (default threshold: **10 MB**), broadcast it to all executors to avoid shuffle entirely.

> **Pro Tip:** Default threshold = 10 MB (`spark.sql.autoBroadcastJoinThreshold`). AQE can also automatically convert sort-merge joins to broadcast joins at runtime.

In [0]:
from pyspark.sql.functions import broadcast

# PySpark — force broadcast join (eliminates shuffle)
# result = df_large.join(broadcast(df_small), "join_key")

# SQL equivalent — hint syntax
# spark.sql("""
#     SELECT /*+ BROADCAST(small_table) */ *
#     FROM large_table
#     JOIN small_table ON large_table.key = small_table.key
# """)

### Solution 4: Handling NULL Keys

NULL values in join or groupBy columns all land on the same partition. Separate them before the join and process independently.

In [0]:
# Separate NULL keys from the join — NULLs never match in inner/left join anyway
# df_nulls = df.filter("join_key IS NULL")
# df_non_nulls = df.filter("join_key IS NOT NULL")

# Process non-null keys (regular join), handle nulls separately
# result = df_non_nulls.join(df_other, "join_key")

### Skew Solutions — Summary

| Problem | Solution | When to Use |
|---------|----------|-------------|
| Skewed join keys | AQE (automatic) | First attempt — usually sufficient |
| Extreme skew in joins | Salting | AQE insufficient, 100:1+ key imbalance |
| Small table join | Broadcast join | One table < 10 MB (adjustable) |
| NULL keys in join/groupBy | Filter + handle separately | Many NULLs in join/group column |
| Skewed groupBy | `repartition()` + salting | One group dominates others |
| Uneven file sizes | Auto Loader / OPTIMIZE | Ingestion or post-load compaction |
| Time partition skew | Liquid Clustering | Replace static partitioning |

> **Pro Tip:** AQE handles most cases automatically. Python UDFs are slower than SQL UDFs due to serialization. Higher-order functions preferred over EXPLODE for arrays.

## Deletion Vectors

Faster DELETE, UPDATE, and MERGE operations by marking rows as deleted in a separate bitmap file instead of rewriting entire Parquet files.

**Theoretical Introduction:**

Deletion Vectors are a storage optimization feature in Delta Lake that improves DELETE, UPDATE, and MERGE performance.

**How it works:**
- Instead of rewriting entire data files on DELETE/UPDATE, Delta Lake marks rows as deleted in a separate **deletion vector file**
- The actual data files remain unchanged until next OPTIMIZE or REORG
- Reads automatically filter out deleted rows using the deletion vector

<img src="../../../assets/images/3358ea8efef949f09710fd4423728a31.png" width="800">

| Aspect | Without Deletion Vectors | With Deletion Vectors |
|--------|------------------------|----------------------|
| DELETE speed | Slow (rewrite files) | Fast (mark in vector) |
| Storage during DELETE | Temporary 2x | Minimal overhead |
| Read performance | Normal | Slight overhead (filter) |
| Cleanup | Automatic | Needs REORG TABLE ... APPLY |

```sql
-- Enable deletion vectors on a table
ALTER TABLE my_table SET TBLPROPERTIES ('delta.enableDeletionVectors' = true);

-- Purge deletion vectors (rewrite files)
REORG TABLE my_table APPLY (PURGE);
```

**Pro Tip:** Deletion Vectors are enabled by default on new tables in Databricks. They make DML operations faster but may slightly increase read overhead until REORG is run.

## Predictive Optimization

Automatic maintenance feature that uses ML to schedule OPTIMIZE and VACUUM — eliminating manual table maintenance for Unity Catalog managed tables.

![ae9f6138b2354c91a096cfd314482a](../../../assets/images/training_2026/day2/ae9f6138b2354c91a096cfd314482ac4.webp "../../../assets/images/training_2026/day2/ae9f6138b2354c91a096cfd314482ac4.webp")

**Key Facts (2025-2026):**

- **Enabled by default** since May 2025 for all Unity Catalog managed tables
- Automatically runs OPTIMIZE (file compaction) when small files accumulate
- Automatically runs VACUUM when orphaned files need cleanup
- Managed at catalog, schema, or table level
- No manual scheduling needed — Databricks decides when to run based on table health metrics

```sql
-- Enable at schema level (already default for new schemas since May 2025)
ALTER SCHEMA my_catalog.my_schema
ENABLE PREDICTIVE OPTIMIZATION;

-- Disable for a specific table (e.g., staging tables)
ALTER TABLE my_catalog.my_schema.staging_temp
DISABLE PREDICTIVE OPTIMIZATION;

-- Check optimization history
SELECT * FROM system.storage.predictive_optimization_operations_history
WHERE catalog_name = 'my_catalog'
ORDER BY timestamp DESC;
```

| Feature | Manual Maintenance | Predictive Optimization |
|---------|-------------------|------------------------|
| OPTIMIZE | Run manually / schedule | Automatic |
| VACUUM | Run manually / schedule | Automatic |
| Tuning | DBA responsibility | ML-based decisions |
| Cost | Compute cost on schedule | Pay per optimization |
| Default | Must configure | Enabled by default (May 2025+) |

> **Pro Tip:** Predictive Optimization requires Unity Catalog managed tables. For staging/temporary tables that are dropped frequently, consider disabling it to avoid unnecessary optimization runs.

## Summary

| Topic | Key Takeaway |
|---|---|
| **OPTIMIZE** | Compacts small files. Run after streaming/frequent inserts |
| **Partitioning** | Low-cardinality columns, partition > 1GB |
| **Z-ORDER** | Co-locates data for data skipping on filter columns |
| **Liquid Clustering** | Modern replacement — incremental, OPTIMIZE still needed but faster |
| **Data Skew** | AQE handles most cases. Salting, broadcast for extreme skew |
| **Deletion Vectors** | Fast DELETE/UPDATE via marking instead of rewriting files |
| **Predictive Optimization** | Automatic OPTIMIZE + VACUUM for UC managed tables |

### Quick Reference

| Operation | Command |
|---|---|
| Optimize | `OPTIMIZE table` |
| Z-ORDER | `OPTIMIZE table ZORDER BY (col)` |
| Vacuum | `VACUUM table RETAIN X HOURS` |
| Deletion Vectors | `ALTER TABLE SET TBLPROPERTIES ('delta.enableDeletionVectors' = true)` |
| Purge DVs | `REORG TABLE ... APPLY (PURGE)` |

> **Note:** Change Data Feed (CDF) has been moved to **M03** (fundamentals) and **M05** (incremental ETL usage).

## Cleanup

Remove demo tables and temporary data created during this module.

In [0]:
# Optional test resource cleanup
# NOTE: Run only if you want to delete all created data

# Tables to clean up:
cleanup_tables = [
    "small_files_demo",
    "sales_zorder_demo",
    "sales_liquid_clustering"
]

In [0]:
# Uncomment below to execute cleanup:
# for table in cleanup_tables:
#     spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.{table}")
#     print(f"Dropped: {table}")

# spark.sql("DROP VIEW IF EXISTS customer_updates")
# spark.catalog.clearCache()

# print("All resources cleaned up!")

← [02 — Delta Lake Advanced](02_delta_advanced_demo.ipynb) | **[ README](../../../README.md)** | [04 — Cost & Security](04_cost_security_demo.ipynb) →